The objective of this project is to develop a Recurrent Neural Network
RNN model to automatically classify news articles into different categories using the AG News dataset. The project involves preprocessing text data, converting words into numerical representations through tokenization, padding sequences to a fixed length, and training an RNN model to learn patterns in news articles. The trained model is evaluated based on its classification accuracy to measure its performance in predicting the correct news category.

In [1]:

!pip install datasets

# Step 1: Install and import all the required libraries
These libraries help in loading the dataset, preprocessing text,
building the RNN model, and evaluating its performance.

In [2]:
from datasets import load_dataset
# Use the full namespace/dataset format
dataset = load_dataset("wangrongsheng/ag_news", split="train")

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [3]:
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

# Step 2: Clean the text data
 Convert all text to lowercase and remove numbers,
punctuation, special characters, and extra spaces.

In [4]:
import re

def clean_my_text(text):
    text = text.lower()                   # Convert to Lowercase
    text = re.sub(r'[^a-z\s]', '', text)  # Delete symbols and numbers
    text = re.sub(r'\s+', ' ', text)      # Remove the extra spaces
    return text.strip()

# Cleaned Dataset
clean_dataset = dataset.map(lambda x: {"text": clean_my_text(x["text"])})

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

# Step 3: Tokenize the text
 Create a vocabulary by assigning a unique integer
 to every unique word and convert text into sequences of numbers.

In [5]:
# Step 3: Tokenize the Strings into Numbers

from tensorflow.keras.preprocessing.text import Tokenizer

# Get all cleaned texts
texts = [sample["text"] for sample in clean_dataset]

# Create Tokenizer
tokenizer = Tokenizer()

# Create the vocabulary (word -> integer)
tokenizer.fit_on_texts(texts)

# Convert text into integer sequences
sequences = tokenizer.texts_to_sequences(texts)

# Vocabulary lookup dictionary
word_index = tokenizer.word_index

print("Vocabulary Size:", len(word_index))
print("First 10 words in vocabulary:")
print(list(word_index.items())[:10])

print("\nFirst text:")
print(texts[0])

print("\nTokenized:")
print(sequences[0])

Vocabulary Size: 91343
First 10 words in vocabulary:
[('the', 1), ('to', 2), ('a', 3), ('of', 4), ('in', 5), ('and', 6), ('on', 7), ('for', 8), ('s', 9), ('that', 10)]

First text:
wall st bears claw back into the black reuters reuters shortsellers wall streets dwindlingband of ultracynics are seeing green again

Tokenized:
[391, 324, 1525, 14260, 99, 54, 1, 812, 23, 23, 38863, 391, 1988, 50537, 4, 38864, 34, 3893, 737, 295]


# Step 4: Pad and truncate sequences
Make every text sequence the same length (50 words)
by adding zeros or removing extra words.

In [6]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
# Set maximum sequence length
max_length = 50
# Pad shorter sequences and truncate longer ones
padded_sequences = pad_sequences(
    sequences,
    maxlen=max_length,
    padding='post',      # Add zeros at the end
    truncating='post'    # Remove extra words from the end
)
# Display the result
print("Shape:", padded_sequences.shape)
print("\nFirst Padded Sequence:")
print(padded_sequences[0])

Shape: (120000, 50)

First Padded Sequence:
[  391   324  1525 14260    99    54     1   812    23    23 38863   391
  1988 50537     4 38864    34  3893   737   295     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0]


# Step 5: Convert Labels to Categorical Vectors


  

In [7]:
labels = clean_dataset["label"]

In [8]:
# Convert labels to a NumPy array
# This converts each class label into a binary vector
# so it can be used for multi-class classification.
labels = np.array(labels)

# One-hot encode the labels
categorical_labels = to_categorical(labels)

print(categorical_labels.shape)
print(categorical_labels[:5])

(120000, 4)
[[0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]]


# STEP 6: Build the RNN model

In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

# Define the RNN model
model = Sequential()
# 1. Embedding Layer
model.add(Embedding(input_dim=len(tokenizer.word_index)+1,output_dim=128,input_length=50))
# 2. SimpleRNN Layer
model.add(SimpleRNN(64))
# 3. Dense Output Layer
model.add(Dense(4, activation='softmax'))
# Display model architecture
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

# Step 10: Evaluate the model
Measure the model's loss and accuracy
 on the dataset after training.

In [15]:
from tensorflow.keras.optimizers import Adam
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    padded_sequences,
    categorical_labels,
    epochs=5,
    batch_size=32,
    valid ation_split=0.2
)

Epoch 1/5
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 511s 169ms/step - accuracy: 0.9580 - loss: 0.1449 - val_accuracy: 0.8352 - val_loss: 0.5487
Epoch 2/5
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 506s 169ms/step - accuracy: 0.9623 - loss: 0.1313 - val_accuracy: 0.8645 - val_loss: 0.5122
Epoch 3/5
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 504s 168ms/step - accuracy: 0.9653 - loss: 0.1213 - val_accuracy: 0.8474 - val_loss: 0.5795
Epoch 4/5
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 556s 166ms/step - accuracy: 0.9608 - loss: 0.1383 - val_accuracy: 0.8525 - val_loss: 0.5577
Epoch 5/5
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 517s 171ms/step - accuracy: 0.9618 - loss: 0.1331 - val_accuracy: 0.8504 - val_loss: 0.5073


# Step 11: Display the final accuracy
 Print the loss and accuracy obtained by the trained model.

In [ ]:
loss, accuracy = model.evaluate(
    padded_sequences,
    categorical_labels,
    verbose=1
)
print("Loss:", loss)
print("Accuracy:", accuracy)

3561/3750 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9673 - loss: 0.1106